# Lilylet dataset visualization

This notebook loads the packed Lilylet dataset described by `configs/lilylet-notagen-data.yaml`
through `starry.utils.dataset_factory.loadDataset`, and prints a readable view of the dataset
items and one collated training batch.

Unlike `lilylet_patchifier_visualization.ipynb` (which patchifies raw `.lyl` text on the fly),
this notebook consumes the pre-packed `.pt` artifact exactly as the trainer does — split
filtering, prompt dropout, supervision boundary, and batch collation all come from the config.

In [1]:
from pathlib import Path
import os
import sys

# Notebook is stored under deep-starry/tests. Add repo root to sys.path.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'starry').is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from starry.utils.config import Configuration
from starry.utils.dataset_factory import loadDataset
from starry.lilylet.data.patchifier import LilyletTokenizer

CONFIG_PATH = REPO_ROOT / 'configs' / 'lilylet-notagen-data.yaml'
# data.root in the config is relative to DATA_DIR; the packed artifact lives under
# ~/data/lilylet/patches. Set the DATA_DIR env var to override.
DATA_DIR = os.environ.get('DATA_DIR') or str(Path.home() / 'data' / 'lilylet' / 'patches')

print('repo:', REPO_ROOT)
print('config:', CONFIG_PATH)
print('data dir:', DATA_DIR)

repo: /home/camus/work/deep-starry
config: /home/camus/work/deep-starry/configs/lilylet-notagen-data.yaml
data dir: /home/camus/data/lilylet/patches/


In [2]:
config = Configuration.createOrLoad(str(CONFIG_PATH))

print('data.type:  ', config['data.type'])
print('data.root:  ', config['data.root'])
print('data.splits:', config['data.splits'])
print('batch_size: ', config['data.batch_size'])
print('tokenizer:  ', config['data.args']['tokenizer_path'])

# loadDataset returns one DataLoader per split (here: train, val).
loaders = loadDataset(config, data_dir=DATA_DIR, device='cpu')
train_loader, val_loader = loaders
train_set = train_loader.dataset
val_set = val_loader.dataset

print()
print('splits:            ', len(loaders))
print('train items:       ', len(train_set))
print('val items:         ', len(val_set))
print('total items (store):', len(train_set.store))

data.type:   LilyletPatchy
data.root:   notagenx-from-abc-20260617.pt
data.splits: *0..8/10:9/10
batch_size:  2
tokenizer:   assets/lilylet-tokenizer.json

splits:             2
train items:        194
val items:          21
total items (store): 215


## Token rendering helpers

Build an id→token map from the tokenizer so packed patch ids can be shown as readable
tokens (same convention as the patchifier notebook: `\n`/`\t` escaped, space shown as `·`,
padding as `<pad>`).

In [3]:
TOKENIZER_PATH = REPO_ROOT / config['data.args']['tokenizer_path']
tokenizer = LilyletTokenizer(str(TOKENIZER_PATH))

id_to_token = {entry['id']: entry['token'] for entry in tokenizer.vocab}

def display_token(token_id: int) -> str:
    token = id_to_token.get(token_id)
    if token is None:
        return f'<id:{token_id}>'
    if token == '\n':
        return r'\n'
    if token == '\t':
        return r'\t'
    if token == ' ':
        return '·'
    if token_id == tokenizer.pad_id:
        return '<pad>'
    return token

def render_patch(row) -> str:
    return ' | '.join(display_token(int(x)) for x in row.tolist())

print('vocab size:', len(id_to_token))
print('pad/bos/eos/unknown ids:', tokenizer.pad_id, tokenizer.bos_id, tokenizer.eos_id, tokenizer.unknown_id)

vocab size: 256
pad/bos/eos/unknown ids: 0 1 2 3


## Inspect a single dataset item

Each item is `(patches, mask, boundary)`:
- `patches`: `[num_patches, patch_size]` token-id grid.
- `mask`: attention mask, all-ones per item (real padding is added only at batch time).
- `boundary`: index of the `<bos>` patch — everything up to and including it is prompt/context;
  supervision (prediction targets) begins after it.

In [4]:
ITEM_INDEX = 0

In [9]:
ITEM_INDEX += 1
patches, mask, boundary = train_set[ITEM_INDEX]

print('item index:      ', ITEM_INDEX)
print('patches shape:   ', tuple(patches.shape))
print('mask shape:      ', tuple(mask.shape), '(sum =', int(mask.sum()), ')')
print('supervision boundary:', boundary, '(patches 0..%d are prompt/context)' % boundary)
print()

MAX_PATCHES = 128
for i in range(min(MAX_PATCHES, patches.shape[0])):
    tag = ' <-- boundary (<bos>)' if i == boundary else ('  [prompt]' if i < boundary else '')
    print(f'patch {i:04d}{tag}')
    print('  tokens:', render_patch(patches[i]))
if patches.shape[0] > MAX_PATCHES:
    print(f'... ({patches.shape[0] - MAX_PATCHES} more patches)')

item index:       2
patches shape:    (541, 16)
mask shape:       (541,) (sum = 541 )
supervision boundary: 3 (patches 0..3 are prompt/context)

patch 0000  [prompt]
  tokens: % | C | l | a | ss | i | c | a | l | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0001  [prompt]
  tokens: % | M | o | z | a | r | t | , | · | W | o | l | f | g | a | n
patch 0002  [prompt]
  tokens: g | · | A | ma | d | e | u | s | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0003 <-- boundary (<bos>)
  tokens: <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <eos>
patch 0004
  tokens: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0005
  tokens: [ | measures | · | " | 2 | * | [ | 1 | . | . | 1 | 7 | ] | , | · | 2
patch 0006
  tokens: * | [ | 1 | 8 | . | . | 3 | 4 | ] | , | · | 2 | * | [ | 3 | 5
patch 0007
  tokens: . | . | 4 | 3 | ] | , | · | 2 | * | [ | 4 | 4

## Compact multi-item overview

First-patch preview of several items across the training split, to eyeball metadata/prompt
headers and item lengths.

In [6]:
N_ITEMS = min(12, len(train_set))
for idx in range(N_ITEMS):
    p, m, b = train_set[idx]
    head = render_patch(p[min(b + 1, p.shape[0] - 1)]) if p.shape[0] else '(empty)'
    print(f'item {idx:04d}  patches={p.shape[0]:4d}  boundary={b:3d}  first-body: {head}')

item 0000  patches= 348  boundary=  3  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
item 0001  patches= 527  boundary=  3  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
item 0002  patches= 542  boundary=  4  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
item 0003  patches= 714  boundary=  2  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
item 0004  patches= 319  boundary=  4  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
item 0005  patches= 441  boundary=  4  first-body: [ | staves | · | " | , | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad>
item 0006  patches= 518  boundary=  4  first-body: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
ite

## Collated training batch

Pull one batch through the DataLoader's `collateBatch` — this is exactly what the model
receives. `input_patches` is padded to the longest item in the batch; `input_masks` marks real
patches; `input_targets` additionally zeroes the prompt + `<bos>` boundary so those positions
are attended but never used as prediction targets.

In [7]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(f'{key:15s} shape={tuple(value.shape)} dtype={value.dtype}')

print()
b0_patches = batch['input_patches'][0]
b0_mask = batch['input_masks'][0]
b0_target = batch['input_targets'][0]
print('batch item 0: real patches =', int(b0_mask.sum()), '/', b0_mask.shape[0],
      '| supervised patches =', int(b0_target.sum()))
print()

SHOW = min(20, b0_patches.shape[0])
for i in range(SHOW):
    flags = ('M' if int(b0_mask[i]) else '-') + ('T' if int(b0_target[i]) else '-')
    print(f'patch {i:04d} [{flags}]  {render_patch(b0_patches[i])}')

input_patches   shape=(2, 526, 16) dtype=torch.int64
input_masks     shape=(2, 526) dtype=torch.int64
input_targets   shape=(2, 526) dtype=torch.int64

batch item 0: real patches = 349 / 526 | supervised patches = 344

patch 0000 [M-]  % | B | a | r | o | q | u | e | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0001 [M-]  % | B | a | c | h | , | · | J | o | h | a | n | n | · | S | e
patch 0002 [M-]  b | a | s | t | i | a | n | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0003 [M-]  % | K | e | y | b | o | a | r | d | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0004 [M-]  <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <bos> | <eos>
patch 0005 [MT]  [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
patch 0006 [MT]  [ | instrument | - | 1 | - | 2 | · | " | H | a | r | p | " | ] | \n | <eos>
patch 0007 [MT]  [ | r | : | 0 | 

## Prompt-dropout variant (`args_variant` 1)

The config defines `args_variant.1` with `prompt_dropout: 0`. The trainer can load a split with a
variant applied; here we load the val split (index 1) which the config maps to `9/10`. This shows
how `loadDataset` respects `data.splits` and per-split arg overrides.

In [8]:
# val_set is the second split; confirm its args resolved from the config.
print('val split filter:', config['data.splits'].split(':')[1])
print('val prompt_dropout:', val_set.prompt_dropout)
print('train prompt_dropout:', train_set.prompt_dropout)

vp, vm, vb = val_set[0]
print()
print('val item 0 patches shape:', tuple(vp.shape), '| boundary:', vb)
print('first body patch:', render_patch(vp[min(vb + 1, vp.shape[0] - 1)]))

val split filter: 9/10
val prompt_dropout: 0
train prompt_dropout: 0.3

val item 0 patches shape: (743, 16) | boundary: 3
first body patch: [ | staves | · | " | { | - | } | " | ] | \n | <eos> | <pad> | <pad> | <pad> | <pad> | <pad>
